In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib as mpl 
import sys
from wave_approx import *

%load_ext autoreload
%autoreload 2

/home/wd/repo/spont_approx/wave_approx.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
quad = False
rLTD = 1.0
r0 = 6
rx = 28
l = 8
d = 3
v = 3.2
k = 0.114*0.017
A = 0.01
tau_m = 0.02
tau = 1
gL = 40
nT = 20
nt = 8000
w0 = 0.1
cap = 12
theme = 'gLs'
shape = 'circle'
match shape:
    case 'circle':
        half_func = half_circle_height
        iint_func = iint_circle_chord
        int_func = int_circle_chord
        if quad:
            func = half_circle_height
        else:
            func = iint_circle_chord
    case 'square':
        half_func = half_square
        iint_func = iint_square
        int_func = int_square
        if quad:
            func = half_square
        else:
            func = iint_square
 

In [ ]:
#gL_range = [10, 20, 40]
#l_range = [4, 8, 16]
l_range = [16]
n = len(l_range)
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
shape = 'circle'
wt = []
for l in l_range:
    wt1, fr1, rbar1 = iter_sweep('exact1', nT = nT, w0 = w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt0, fr0, rbar0 = iter_sweep('exact0', nT = nT, w0 = w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = rLTD, r0 = r0, rx = rx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    wt.append(wt1)
    nx = wt1.shape[0]
    #ax[0].plot(np.arange(nx), wt0[:,-1], label = 'exact0')
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact1')
    ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    #ax[1].plot(np.arange(nT), [np.max(fr0[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact0')
    ax[1].plot(np.arange(nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact1')
    ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    ax[1].legend()
    fig.savefig(f'l-{l}_traces.png')
    fig.savefig(f'l-{l}_traces.svg')

T = 16.017, dt = 2.002e-03, each LGN activated for 11.7%
max weight possible: 1.200e+00
using method exact1
5/20... 

In [ ]:
y2 = np.array([0, 0])
print(y2)
j = 0
w = wt[j][:,-1]
uniform_W = False
#w = np.zeros(wt2.shape[0]) + w0
print(w.min(),w.max())
L = 2*l_range[j]*np.sqrt(2) + 2*d
T = L/v
dt = 1e-3
nt = int(T/dt)+2
print(f'nt = {nt}')
t = np.arange(nt)*dt
n = 10
p = np.linspace(0.05,0.95,n)
r = np.linspace(0, 30, 30)
rbar = np.linspace(0, 30, 30)
# solve approx2, plot trace
fig = plt.figure(figsize = (14/5*n,10), dpi = 200)
# phase plane
k_prime = k*A
a = k_prime*rx*rx*int_func(T*p, v=v, d=d, l=l_range[j])
nx = w.size
x = np.linspace(0, 2*l_range[j], nx)
print(f'k = {k_prime:.3e}, coef_0 = {k_prime*rx*rx:.3e}')
for i in range(n):
    ax = fig.add_subplot(4,n,i+1)
    phase_plane(get_dF2(w, uniform_W, half_func, iint_func, T*p[i], k = k_prime, r0 = r0, l = l_range[j], d = d, v = v, rx = rx, tau = tau, gL = gL), r, rbar, ax = ax)
    _r = np.linspace(0,r[-1], 1000)
    _rbar, ro = nullcline2(_r, w, uniform_W, half_func, iint_func, T*p[i], k_prime, l_range[j], d, v, r0, rx, gL)
    ax.plot(_r, _rbar, 'g', lw = 1, alpha = 1, zorder = 0)
    if ro is not None:
        ax.plot(ro, 0, 'og', fillstyle = 'none', alpha = 1, zorder = 0)
    ax.plot(_r, _r, lw = 1, alpha = 1, zorder = 0)
    ax.plot(2/3*r0+np.zeros_like(rbar), rbar, 'k', lw = 1, alpha = 0.7, zorder = 0)
    if a[i] > 1/r0:
        ax.plot(r0 + np.sqrt(r0*r0-1/a[i]) + np.zeros_like(rbar), rbar, ':k', lw = 1, alpha = 0.7, zorder = 0)
        ax.plot(r0 - np.sqrt(r0*r0-1/a[i]) + np.zeros_like(rbar), rbar, ':k', lw = 1, alpha = 0.7, zorder = 0)
    x_bot, x_top = get_xrange(T*p[i],v,d,l_range[j])
    ax.set_title(f'{p[i]*100:.0f}% T\n a = {a[i]:.3e}\n base = {2*v*rx*(np.interp(x_top, x, w)*half_func(x_top, l_range[j]) - np.interp(x_bot, x, w)*half_func(x_bot, l_range[j])):.1f}')
    ax.set_xlim(r[0], r[-1])
    ax.set_ylim(rbar[0], rbar[-1])
    if i == 0:
        ax.set_ylabel('avg. fr (Hz)')
    ax.set_xlabel('fr (Hz)')
    
ax = fig.add_subplot(4,1,2)

sol2 = solve_ivp(get_approx2(w, uniform_W, half_func, iint_func, k = k_prime, l = l_range[j], d = d, v = v, r0 = r0, rx = rx, gL = gL, tau = tau), t, y2)
    
ax.plot(sol2.t, sol2.y[0,:], 'b', label = 'fr')
ax.plot(sol2.t, sol2.y[1,:], 'g', label = 'avg. fr', alpha = 0.7)
print(f'coef_1 = {2*v*rx*w0}')
ax.plot(sol2.t, np.power(sol2.y[1,:],2)/r0, ':r', label = 'thres. fr')

ax.set_ylabel('f Hz')
ax.legend(loc = 'upper left')
ax = ax.twinx()
ax.plot(T*p, a, '*k', ms = 4)
ax.plot(t, k*rx*rx*int_func(t, v=v, d=d, l=l_range[j]), ':k', alpha = 0.5, label = 'input')
ax.legend()
ax.set_ylabel('\delta W/s')

ax = fig.add_subplot(4,1,3)
x_bot, x_top = get_xrange(t,v,d,l_range[j])
ax.plot(t, 2*v*rx*(half_func(x_top, l_range[j])*np.interp(x_top,x,w) - half_func(x_bot, l_range[j])*np.interp(x_bot,x,w)))

ax0 = fig.add_subplot(4,2,7)
ax1 = fig.add_subplot(4,2,8)
# plot W over time
w2 = next_W(t,w,sol2.y[0,:],sol2.y[1,:], k,l_range[j],d,v,rx,r0, end_only = False)
show_w_over_t(w2, dt, 't (s)', 'W', l_range[j], ax = [ax0, ax1])
fig.savefig(f'phase_trace.png')
fig.savefig(f'phase_trace.svg')

In [ ]:
# rLTD
l = 8
rLTD_range = [0.1,0.2,0.4,0.8,1.0]
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
A = 0.015
shape = 'circle'
w0 = 0.05
nT = 5
l = 8
nx = 102
wt = []
fr = []
for _rLTD in rLTD_range:
    wt1, fr1, rbar1 = iter_sweep('exact1', nT = nT, w0 = w0, l = l, d = d, v = v, k = k, A = A*rLTD, rLTD = _rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt0, fr0, rbar0 = iter_sweep('exact0', nT = nT, w0 = w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = _rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    nx = wt1.shape[0]
    #ax[0].plot(np.arange(nx), wt0[:,-1], label = 'exact0')
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact1')
    #ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    #ax[1].plot(np.arange(nT), [np.max(fr0[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact0')
    ax[1].plot(np.arange(nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact1')
    #ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    wt.append(wt1[:,-1])
    fr.append([np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)])
    ax[1].legend()
    fig.savefig(f'rLTD-{_rLTD}_traces.png')
    fig.savefig(f'rLTD-{_rLTD}_traces.svg')

In [ ]:
plt.rcParams['text.usetex'] = True
fig, ax = plt.subplots(1,2, figsize = (7,3), dpi = 200)
n = len(rLTD_range)
lw = 2
for i in range(n):
    ax[0].plot(np.linspace(0,2*l,nx), wt[i], 'b', lw = (i+1)/n * lw, label = rf'$r_{{LTD}}$={rLTD_range[i]}')
    ax[1].plot(np.arange(nT)+1, fr[i], lw = (i+1)/n * lw, label = f'rLTD={rLTD_range[i]}')
ax[0].legend()
ax[0].set_xlabel('visual position (deg)')
ax[0].set_ylabel('weight')
ax[1].set_xlabel('time (\#sweep)')
ax[1].set_ylabel('rate (Hz)')
fig.savefig('rLTD_change.png')
fig.savefig('rLTD_change.svg')

In [ ]:
# w0
l = 8
rLTD = 1.0
w0_range = [0.05,0.1,0.2,0.4]
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
A = 0.025
shape = 'circle'
nT = 10
nx = 102
wt = []
fr = []
for _w0 in w0_range:
    wt1, fr1, rbar1 = iter_sweep('exact1', nT = nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    nx = wt1.shape[0]
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact1')
    #ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    ax[1].plot(np.arange(nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact1')
    wt.append(wt1[:,-1])
    fr.append([np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)])
    #ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    ax[1].legend()
    fig.savefig(f'w0-{_w0}_traces.png')
    fig.savefig(f'w0-{_w0}_traces.svg')

In [ ]:
plt.rcParams['text.usetex'] = False
fig, ax = plt.subplots(1,2, figsize = (7,3), dpi = 200)
n = len(w0_range)
lw = 2
for i in range(n):
    ax[0].plot(np.linspace(0,2*l,nx), wt[i], 'b', lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
    ax[1].plot(np.arange(nT)+1, fr[i], lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
ax[0].legend()
ax[0].set_xlabel('visual position (deg)')
ax[0].set_ylabel('weight')
ax[1].set_xlabel('time (#sweep)')
ax[1].set_ylabel('rate (Hz)')
fig.tight_layout()
fig.savefig('w0_change.png')
fig.savefig('w0_change.svg')

In [ ]:
# w0
l = 8
rLTD = 1.0
w0_range = [0.05,0.1,0.2,0.4]
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
A = 0.025
shape = 'circle'
nT = 10
nx = 102
wt = []
fr = []
for _w0 in w0_range:
    wt1, fr1, rbar1 = iter_sweep('exact2', nT = nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    nx = wt1.shape[0]
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact2')
    #ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    ax[1].plot(np.arange(nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact1')
    wt.append(wt1[:,-1])
    fr.append([np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)])
    #ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    ax[1].legend()
    fig.savefig(f'w0-{_w0}_traces2.png')
    fig.savefig(f'w0-{_w0}_traces2.svg')

In [ ]:
plt.rcParams['text.usetex'] = False
fig, ax = plt.subplots(1,2, figsize = (7,3), dpi = 200)
n = len(w0_range)
lw = 2
for i in range(n):
    ax[0].plot(np.linspace(0,2*l,nx), wt[i], 'b', lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
    ax[1].plot(np.arange(nT)+1, fr[i], lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
ax[0].legend()
ax[0].set_xlabel('visual position (deg)')
ax[0].set_ylabel('weight')
ax[1].set_xlabel('time (#sweep)')
ax[1].set_ylabel('rate (Hz)')
fig.tight_layout()
fig.savefig('w0_change2.png')
fig.savefig('w0_change2.svg')

In [ ]:
# rx
l = 8
rLTD = 1.0
rx_range = [16,18,20,22,24]
w0 = 0.05
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
A = 0.025
shape = 'circle'
nT = 30
nx = 102
wt = []
fr = []
for _rx in rx_range:
    #_nT = int(np.round(nT * np.power(20/_rx,1)))
    _nT = nT
    wt1, fr1, rbar1 = iter_sweep('exact1', nT = _nT, w0 = w0, l = l, d = d, v = v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = _rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    nx = wt1.shape[0]
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact1')
    #ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    ax[1].plot(np.linspace(1, nT, _nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(_nT)], label = 'exact1')
    wt.append(wt1[:,-1])
    fr.append([np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(_nT)])
    #ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    ax[1].legend()

In [ ]:
plt.rcParams['text.usetex'] = True
fig, ax = plt.subplots(1,2, figsize = (7,3), dpi = 200)
n = len(rx_range)
lw = 2
for i in range(n):
    ax[0].plot(np.linspace(0,2*l,nx), wt[i]/np.max(wt[i]), 'b', lw = (i+1)/n * lw, label = rf'$r_{{x}}$={rx_range[i]}')
    ax[1].plot(np.linspace(1, nT, len(fr[i])), fr[i], lw = (i+1)/n * lw, label = f'r_{{x}}={rx_range[i]}')
ax[0].legend()
ax[0].set_xlabel('visual position (deg)')
ax[0].set_ylabel('weight')
ax[1].set_xlabel('time (sweep)')
ax[1].set_ylabel('rate (Hz)')
fig.tight_layout()
fig.savefig('rx_change_long.png')
fig.savefig('rx_change_long.svg')

In [ ]:
# w0
l = 8
rLTD = 1.0
w0 = 0.05
i = 0
fork_ivp = True 
dT = 2
_nt = nt
#for gL in gL_range:
A = 0.025
shape = 'circle'
nT = 10
nx = 102
wt = []
fr = []
v_range = [1.6,3.2,6.4]
for _v in v_range:
    wt1, fr1, rbar1 = iter_sweep('exact1', nT = nT, w0 = w0, l = l, d = d, v = _v, k = k, A = A, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    #wt2, fr2, rbar2 = iter_sweep('approx2', nT = dT*nT, w0 = _w0, l = l, d = d, v = v, k = k, A = A/dT, rLTD = rLTD, r0 = r0, rx = rx, nx = nx, tau = tau, tau_m = tau_m, gL = gL, cap = cap, nt = _nt, plot = True, theme = f'{theme}_{i}', fork_ivp = fork_ivp, shape = shape)
    i += 1
    fig, ax = plt.subplots(1, 2, figsize = (7,3))
    nx = wt1.shape[0]
    ax[0].plot(np.arange(nx), wt1[:,-1], label = 'exact1')
    #ax[0].plot(np.arange(nx), wt2[:,-1], ':', label = 'approx2')
    ax[0].legend()
    ax[1].plot(np.arange(nT), [np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)], label = 'exact1')
    wt.append(wt1[:,-1])
    fr.append([np.max(fr1[i*nt+1:(i+1)*nt+1]) for i in range(nT)])
    #ax[1].plot(np.arange(nT*dT)/dT, [np.max(fr2[i*_nt+1:(i+1)*_nt+1]) for i in range(nT*dT)], label = 'approx2')
    ax[1].legend()
    fig.savefig(f'v-{_v}_traces.png')
    fig.savefig(f'v-{_v}_traces.svg')

In [ ]:
plt.rcParams['text.usetex'] = False
fig, ax = plt.subplots(1,2, figsize = (7,3), dpi = 200)
n = len(w0_range)
lw = 2
for i in range(n):
    ax[0].plot(np.linspace(0,2*l,nx), wt[i], 'b', lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
    ax[1].plot(np.arange(nT)+1, fr[i], lw = (i+1)/n * lw, label = f'w_0={w0_range[i]}')
ax[0].legend()
ax[0].set_xlabel('visual position (deg)')
ax[0].set_ylabel('weight')
ax[1].set_xlabel('time (#sweep)')
ax[1].set_ylabel('rate (Hz)')
fig.tight_layout()
fig.savefig('w0_change.png')
fig.savefig('w0_change.svg')